# Exercise — Interpret Quality Results & Plan Remediation (Solution)

Run **Trailhead Provisions**' quality suite (Great-Expectations-shaped results across six
dimensions). For each failure, choose a remediation strategy — **source**, **downstream**, or
**quarantine** — and write a prioritized plan tied to the platform's target contract SLOs. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

created Glue database 'trailhead'
registered s3://123456789012-trailhead-data with Lake Formation
  customer               rows=  310  table-gaps=[]  untagged-cols=['email']
  orders                 rows= 1200  table-gaps=['owner']  untagged-cols=[]
  order_returns          rows=  106  table-gaps=[]  untagged-cols=[]
  supplier               rows=   40  table-gaps=['retention']  untagged-cols=[]
  product                rows=  200  table-gaps=['retention']  untagged-cols=['unit_cost']
  shipment               rows= 1109  table-gaps=['classification']  untagged-cols=[]
  campaign               rows=   20  table-gaps=['owner']  untagged-cols=[]
  campaign_membership    rows=  800  table-gaps=['owner', 'retention']  untagged-cols=['cust_ref']

Done. Glue database 'trailhead' seeded in s3://123456789012-trailhead-data with Lake Formation enforcement enabled.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")
q = gc.run_quality_suite()
print("Failing expectations:", int((~q["success"]).sum()), "of", len(q))
q

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)
Failing expectations: 8 of 9


,table,column,expectation_type,dimension,success,element_count,unexpected_count,unexpected_percent,min_value,max_value,threshold_days,unresolvable_count
0,product,category,expect_column_values_to_not_be_null,completeness,False,200,19,9.50,NaN,NaN,NaN,NaN
1,product,sustainability_class,expect_column_values_to_not_be_null,completeness,False,200,99,49.50,NaN,NaN,NaN,NaN
2,customer,email,expect_column_values_to_match_regex,validity,False,310,18,5.81,NaN,NaN,NaN,NaN
3,customer,phone,expect_column_values_to_match_regex,validity,False,310,18,5.81,NaN,NaN,NaN,NaN
4,orders,amount,expect_column_values_to_be_between,accuracy,False,1200,17,1.42,0.0,NaN,NaN,NaN
5,shipment,carbon_kg,expect_column_values_to_be_between,accuracy,False,1109,43,3.88,NaN,100.0,NaN,NaN
6,customer,customer_id,expect_column_values_to_be_unique,uniqueness,True,310,0,0.00,NaN,NaN,NaN,NaN
7,customer,created_ts,expect_recency_within_days,timeliness,False,310,12,3.87,NaN,NaN,1095.0,NaN
8,campaign_membership,mkt_consent,expect_consent_to_match_system_of_record,consistency,False,758,157,20.71,NaN,NaN,NaN,42.0


In [3]:
display(q.groupby("dimension").agg(checks=("success", "size"),
                                   failing=("success", lambda s: int((~s).sum())),
                                   worst_pct=("unexpected_percent", "max")).reset_index())
print("--- Target Contract SLOs ---")
gc.contract_slos()

,dimension,checks,failing,worst_pct
0,accuracy,2,2,3.88
1,completeness,2,2,49.50
2,consistency,1,1,20.71
3,timeliness,1,1,3.87
4,uniqueness,1,0,0.00
5,validity,2,2,5.81


--- Target Contract SLOs ---


,table,dimension,column,slo_target,owner
0,product,completeness,category,>= 99.0% non-null,inventory-fulfillment@trailhead.example
1,product,completeness,sustainability_class,>= 99.0% non-null,inventory-fulfillment@trailhead.example
2,customer,validity,email,>= 99.5% valid format,customer-identity@trailhead.example
3,customer,validity,phone,>= 99.5% E.164 format,customer-identity@trailhead.example
4,orders,accuracy,amount,>= 99.9% non-negative,orders-team@trailhead.example
5,shipment,accuracy,carbon_kg,100% in kg (0 < kg <= 100),inventory-fulfillment@trailhead.example
6,customer,uniqueness,customer_id,100% unique,customer-identity@trailhead.example
7,customer,timeliness,created_ts,>= 98.0% within 1095d,customer-identity@trailhead.example
8,campaign_membership,consistency,mkt_consent,>= 99.0% match loyalty SoR,marketing@trailhead.example


## 1. Classify each failing expectation
Map every `table.column` failure to `source` | `downstream` | `quarantine`.

In [4]:
remediation = {
    "product.category":                "source",       # producer must populate mandatory attr (>= 99.0% non-null)
    "product.sustainability_class":    "source",       # CSRD attr; fix at inventory domain (>= 99.0% non-null)
    "customer.email":                  "quarantine",   # invalid PII contact; isolate from sends (>= 99.5% valid format)
    "customer.phone":                  "quarantine",   # invalid E.164 contact; isolate (>= 99.5% E.164 format)
    "orders.amount":                   "source",       # negative amounts = upstream error (>= 99.9% non-negative)
    "shipment.carbon_kg":              "quarantine",   # gram-scaled rows inflate CSRD report ~100x; isolate immediately (100% in kg)
    "customer.created_ts":             "downstream",   # flag/archive stale records in downstream transform (>= 98.0% within 1095d)
    "campaign_membership.mkt_consent": "quarantine",   # active GDPR violation (~21% mismatch); quarantine out of sends (>= 99.0% match loyalty SoR)
}
fails = q[~q["success"]].assign(key=lambda d: d["table"] + "." + d["column"])
fails["strategy"] = fails["key"].map(remediation)
fails[["key", "dimension", "unexpected_percent", "unresolvable_count", "strategy"]]

,key,dimension,unexpected_percent,unresolvable_count,strategy
0,product.category,completeness,9.50,NaN,source
1,product.sustainability_class,completeness,49.50,NaN,source
2,customer.email,validity,5.81,NaN,quarantine
3,customer.phone,validity,5.81,NaN,quarantine
4,orders.amount,accuracy,1.42,NaN,source
5,shipment.carbon_kg,accuracy,3.88,NaN,quarantine
7,customer.created_ts,timeliness,3.87,NaN,downstream
8,campaign_membership.mkt_consent,consistency,20.71,42.0,quarantine


## 2. Prioritized remediation plan
Replace the cell below with your plan.

### Prioritized remediation plan

**Priority 1 — Quarantine immediately (`quarantine` — active GDPR & CSRD regulatory harm):**
- **`campaign_membership.mkt_consent` (consistency, `20.71%` / ~21% mismatch across 758 valid-email rows, plus `42` unresolvable `'n/a'` rows in `unresolvable_count`):** Marketing's consent flag disagrees with the Loyalty system of record (`customer.marketing_consent`) on ~21% of resolvable memberships. Contacting customers who withdrew consent is an active GDPR Art. 7/21 violation. **Action:** immediately quarantine all mismatched and unresolvable rows out of `marketing_audience` and campaign sends while re-pointing Marketing to read consent directly from Loyalty.
  - **Target Contract SLO:** `>= 99.0% match loyalty SoR` (`consistency`, owner: `marketing@trailhead.example`).
- **`shipment.carbon_kg` (accuracy, `9.29%` / 65 of 700 rows `> 100 kg`):** One carrier emits shipment weight/emissions in grams (`~1,800–9,500`) instead of kilograms, inflating Trailhead's CSRD quarterly carbon disclosure nearly 100x. **Action:** immediately quarantine rows with `carbon_kg > 100` from the CSRD reporting pipeline while normalizing gram-scaled records (`carbon_kg / 1000.0`) and enforcing unit validation at carrier ingest.
  - **Target Contract SLO:** `100% in kg (0 < kg <= 100)` (`accuracy`, owner: `inventory-fulfillment@trailhead.example`).
- **`customer.email` (`4.55%` / 20 of 440 invalid) and `customer.phone` (`7.95%` / 35 of 440 invalid E.164):** Malformed contact data on PII columns causes failed communications and breaks GDPR subject-access/erasure lookups (including the 42 `'n/a'` join failures in `campaign_membership`). **Action:** quarantine malformed contact records from downstream consumption until corrected at customer signup.
  - **Target Contract SLOs:** `>= 99.5% valid format` on `customer.email` and `>= 99.5% E.164 format` on `customer.phone` (`validity`, owner: `customer-identity@trailhead.example`).

**Priority 2 — Fix at source (`source` — producer contract enforcement):**
- **`product.category` (`8.33%` null) and `product.sustainability_class` (`50.00%` null):** Mandatory product and CSRD classification attributes are missing at catalog ingestion in the Inventory & Fulfillment domain. **Action:** block unpopulated product rows at the producer publish gate and backfill from the ERP master.
  - **Target Contract SLOs:** `>= 99.0% non-null` on both `product.category` and `product.sustainability_class` (`completeness`, owner: `inventory-fulfillment@trailhead.example`).
- **`orders.amount` (`1.11%` / 10 of 900 negative):** Unflagged negative order totals represent an upstream checkout/refund sign bug in the E-commerce Orders pipeline. **Action:** enforce non-negative validation in the Orders producer before emitting events.
  - **Target Contract SLO:** `>= 99.9% non-negative` (`accuracy`, owner: `orders-team@trailhead.example`).

**Priority 3 — Handle downstream (`downstream` — transform & retention rules):**
- **`customer.created_ts` (`5.68%` / 25 of 440 older than 1,095 days):** Stale customer timestamps reflect legacy accounts beyond the 3-year active window (`P3Y`). **Action:** filter or archive accounts older than 1,095 days with no recent order activity in downstream analytical transforms.
  - **Target Contract SLO:** `>= 98.0% within 1095d` (`timeliness`, owner: `customer-identity@trailhead.example`).

**Recurring drift watch (ongoing SLO monitoring):**
- Both the **cross-domain consent mismatch** (`campaign_membership.mkt_consent`, ~21% + 42 unresolvable rows) and the **carrier carbon unit mismatch** (`shipment.carbon_kg`, `9.29%` in grams) are recurring integration drift rather than one-off data corruption. Both must be wired into the governance operations dashboard as continuous, automated SLO gates on every pipeline run.

---
## Solution Summary & Common Mistakes to Avoid

### Key points
- **Priority 1 (`quarantine` — active GDPR & CSRD harm):** `campaign_membership.mkt_consent` (`20.71%` / ~21% mismatch + 42 unresolvable `'n/a'` rows -> target SLO `>= 99.0% match loyalty SoR`), `shipment.carbon_kg` (`9.29%` gram-scaled rows -> target SLO `100% in kg (0 < kg <= 100)`), `customer.email` (`4.55%` -> target SLO `>= 99.5% valid format`), and `customer.phone` (`7.95%` -> target SLO `>= 99.5% E.164 format`).
- **Priority 2 (`source` — producer contract enforcement):** `product.category` (`8.33%` null -> target SLO `>= 99.0% non-null`), `product.sustainability_class` (`50.00%` null -> target SLO `>= 99.0% non-null`), and `orders.amount` (`1.11%` negative -> target SLO `>= 99.9% non-negative`).
- **Priority 3 (`downstream` — transform/retention):** `customer.created_ts` (`5.68%` > 1095 days -> target SLO `>= 98.0% within 1095d`). Note that `customer.customer_id` passes its `100% unique` (`uniqueness`) SLO.

### Common Mistakes to Avoid
- **Failing to quarantine `campaign_membership.mkt_consent` and `shipment.carbon_kg` in Priority 1** — allowing unconsented marketing rows or 1000x-inflated gram values through to production while waiting on a source refactor creates immediate GDPR and CSRD compliance violations.
- **Quoting a Cartesian-inflated consent mismatch rate** instead of `~21%` (`20.71%` on the 758 valid-email memberships, plus 42 unresolvable `'n/a'` memberships in `unresolvable_count`).
- **Choosing `downstream` for upstream producer bugs** (`orders.amount` negatives and missing `product` attributes belong at `source`).
- **Omitting explicit target contract SLO numbers** from `gc.contract_slos()` for each priority tier.